# 3. Model evaluation

Reads what the evaluation scripts wrote. Run these first, from `backend/`:

```
python -m training.evaluate
python -m training.baseline
python -m training.error_analysis
```

In [ ]:
import sys
from pathlib import Path

# Make `app` and `training` importable from the notebooks folder.
BACKEND = Path.cwd().resolve().parent / "backend" if Path.cwd().name == "notebooks" else Path.cwd() / "backend"
sys.path.insert(0, str(BACKEND))

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", 120)

In [ ]:
import json
from app.paths import DOCS_DIR, MODELS_DIR

report = json.loads((MODELS_DIR / "evaluation_report.json").read_text(encoding="utf-8"))
print("generated:", report["generated_at"], "| test domains:", report["domains"])
pd.DataFrame({name: {
    "domains": ", ".join(log["hyperparameters"]["domains"]),
    "epochs": log["hyperparameters"]["epochs"],
    "best epoch": log["best_epoch"],
    "validation score": round(log["best_validation_score"], 4),
    "minutes": log["training_minutes"],
} for name, log in report["training"].items()})

## Headline metrics per test domain

In [ ]:
def headline(results):
    return {
        "aspect F1": results["aspect_extraction"]["f1"],
        "sentiment accuracy": results["aspect_sentiment"]["accuracy"],
        "sentiment macro-F1": results["aspect_sentiment"]["macro"]["f1"],
        "mixed-sentence accuracy": results["aspect_sentiment_by_sentence_type"]["mixed"]["accuracy"],
        "overall macro-F1": results["overall_sentiment"]["macro"]["f1"],
        "end-to-end F1": results["end_to_end"]["f1"],
    }

metrics = pd.DataFrame({scope: headline(r) for scope, r in report["results"].items()}).round(4)
display(metrics)
metrics.T.plot(kind="bar", figsize=(8, 3.8), rot=0, ylim=(0, 1), title="Test-set metrics")
plt.legend(fontsize=7, ncol=2); plt.tight_layout(); plt.show()

## Per-class results and confusion matrix (aspect sentiment)

Neutral is the hardest class.

In [ ]:
sentiment = report["results"]["all"]["aspect_sentiment"]
display(pd.DataFrame(sentiment["per_class"]).T.round(4))
confusion = sentiment["confusion_matrix"]
pd.DataFrame(confusion["matrix"], index=[f"true {l}" for l in confusion["labels"]],
             columns=[f"pred {l}" for l in confusion["labels"]])

In [ ]:
from IPython.display import Image, display as show
for name in ("sentiment_confusion_matrix.png", "overall_confusion_matrix.png"):
    show(Image(filename=str(DOCS_DIR / "figures" / name), width=420))

## What BERT adds over a rule-and-lexicon baseline

In [ ]:
baseline = json.loads((MODELS_DIR / "baseline_report.json").read_text(encoding="utf-8"))
paths = {"aspect F1": ("aspect_extraction", "f1"), "sentiment macro-F1": ("aspect_sentiment", "macro", "f1"),
         "overall macro-F1": ("overall_sentiment", "macro", "f1"), "end-to-end F1": ("end_to_end", "f1")}

def pick(results, path):
    for key in path:
        results = results[key]
    return results

comparison = pd.DataFrame({
    "baseline": {n: pick(baseline["results"]["all"], p) for n, p in paths.items()},
    "BERT": {n: pick(report["results"]["all"], p) for n, p in paths.items()},
}).round(4)
comparison["difference"] = (comparison.BERT - comparison.baseline).round(4)
display(comparison)
comparison[["baseline", "BERT"]].plot(kind="bar", figsize=(7, 3.5), rot=0, ylim=(0, 1), title="Baseline vs. BERT, both domains")
plt.tight_layout(); plt.show()

## Error analysis and transfer

The full tables are in `docs/error_analysis.md` and `docs/transfer.md`.

In [ ]:
from IPython.display import Markdown
for name in ("error_analysis.md", "transfer.md"):
    path = DOCS_DIR / name
    text = path.read_text(encoding="utf-8") if path.exists() else f"`{name}` not found: run `python -m training.error_analysis`."
    show(Markdown(text))